# Stable Baselines3 Training - CarRacing with 1 Agent

In [ ]:
from pathlib import Path
from stable_baselines3 import SAC, DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack, VecCheckNan, VecTransposeImage
)
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback, CallbackList
import importlib
importlib.import_module("gym_multi_car_racing")

SEED = 42
run_dir = Path("experiments/dqn_multicarracing")
run_dir.mkdir(parents=True, exist_ok=True)

env_kwargs = {"continuous": False, "num_agents": 1}

# Train env
train_env = make_vec_env(
    "MultiCarRacing-v1",
    n_envs=4,
    seed=SEED,
    vec_env_cls=DummyVecEnv,
    monitor_dir=str(run_dir / "monitor_train"),
    env_kwargs=env_kwargs
)
train_env = VecMonitor(train_env)
train_env = VecFrameStack(train_env, n_stack=4)
train_env = VecTransposeImage(train_env)
train_env = VecNormalize(train_env, norm_obs=False, norm_reward=True, clip_reward=10.0)

# Eval env (separate)
eval_env = make_vec_env(
    "MultiCarRacing-v1",
    n_envs=1,
    seed=SEED + 1000,
    vec_env_cls=DummyVecEnv,
    monitor_dir=str(run_dir / "monitor_eval"),
    env_kwargs=env_kwargs
)
eval_env = VecMonitor(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)
eval_env = VecTransposeImage(eval_env)
eval_env = VecNormalize(eval_env, norm_obs=False, norm_reward=False, training=False)

eval_cb = EvalCallback(
    eval_env=eval_env,
    best_model_save_path=str(run_dir / "best_model"),
    log_path=str(run_dir / "eval_logs"),
    eval_freq=10_000,
    n_eval_episodes=5,
    deterministic=True,
)

ckpt_cb = CheckpointCallback(
    save_freq=25_000,
    save_path=str(run_dir / "checkpoints"),
    name_prefix="dqn",
)

model = DQN(
    "CnnPolicy",
    train_env,
    seed=SEED,
    device="mps",
    verbose=1,
    tensorboard_log=str(run_dir / "tb"),
)

model.learn(total_timesteps=100_0000, callback=CallbackList([eval_cb, ckpt_cb]), progress_bar=True)
model.save(str(run_dir / "final_model"))
train_env.save(str(run_dir / "vecnormalize.pkl"))

# Evaluation

In [ ]:
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack, VecTransposeImage
)
import importlib
importlib.import_module("gym_multi_car_racing")

run_dir = Path("experiments/dqn_multicarracing")
env_kwargs = {"continuous": False, "num_agents": 1}

# Build the SAME pre-normalize wrapper stack as training
eval_env = make_vec_env(
    "MultiCarRacing-v1",
    n_envs=1,
    vec_env_cls=DummyVecEnv,
    env_kwargs=env_kwargs,
)
eval_env = VecMonitor(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)
eval_env = VecTransposeImage(eval_env)

# Now load normalization stats (shapes match)
eval_env = VecNormalize.load(str(run_dir / "vecnormalize.pkl"), eval_env)
eval_env.training = False
eval_env.norm_reward = False

model = DQN.load(str(run_dir / "best_model" / "best_model.zip"), env=eval_env, device="mps")
mean_r, std_r = evaluate_policy(model, eval_env, n_eval_episodes=20, deterministic=True)
print(mean_r, std_r)

# Testing

In [ ]:
from pathlib import Path
from stable_baselines3 import DQN
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import (
    DummyVecEnv, VecMonitor, VecNormalize, VecFrameStack, VecTransposeImage
)
import importlib
importlib.import_module("gym_multi_car_racing")

run_dir = Path("experiments/dqn_multicarracing")
env_kwargs = {"continuous": False, "num_agents": 1}

# Build the SAME pre-normalize wrapper stack as training
eval_env = make_vec_env(
    "MultiCarRacing-v1",
    n_envs=1,
    vec_env_cls=DummyVecEnv,
    env_kwargs=env_kwargs,
)
eval_env = VecMonitor(eval_env)
eval_env = VecFrameStack(eval_env, n_stack=4)
eval_env = VecTransposeImage(eval_env)

# Now load normalization stats (shapes match)
eval_env = VecNormalize.load(str(run_dir / "vecnormalize.pkl"), eval_env)
eval_env.training = False
eval_env.norm_reward = False

model = DQN.load(str(run_dir / "best_model" / "best_model.zip"), env=eval_env, device="mps")

vec_env = model.get_env()
obs = vec_env.reset()
done = False
while not done:
    action, _state = model.predict(obs, deterministic=True)
    obs, reward, done, info = vec_env.step(action)
    vec_env.render("human")